In [1]:
import os
import json
import re
import sqlite3
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Target Configuration ───────────────────────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/github/persverscomp") 
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = WORKSPACE_DIR / "corpus_alignment_grid.db"
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

WORK_REGISTRY = {
    "tlg0003.tlg001": {
        "textgroup": "tlg0003",
        "work": "tlg001",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.xml", "label": "Greek (H. S. Jones, 1942)", "class": "greek-text"},
            "1st1K-eng1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng1.xml", "label": "English (C. F. Smith, 1919)", "class": "english-text"},
            "1st1K-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng2.xml", "label": "English (Henry Dale, 1851)", "class": "english-text"},
            "perseus-eng4": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng4.xml", "label": "English (Thomas Hobbes, 1843)", "class": "english-text"},
            "perseus-eng6": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng6.xml", "label": "English (R. Crawley, 1914)", "class": "english-text"},
            "1st1k-fre1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-fre1.xml", "label": "French (E. Bétant, 1863)", "class": "french-text"},
            "1st1K-fre2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-fre2.xml", "label": "French (M. C. Zévort, 1852)", "class": "french-text"},
            "1st1K-ger1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger1.xml", "label": "German (H. Gürsching, 1856)", "class": "german-text"},
            "1st1K-ger2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger2.xml", "label": "German (A. Wahrmund, 1864)", "class": "german-text"},
            "1st1K-ger3": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger3.xml", "label": "German (Theodor Braun, 1917)", "class": "german-text"},
            "1st1K-ger4": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ger4.xml", "label": "German (Rudolf G. Binding, 1937)", "class": "german-text"},
            "1st1K-ita1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-ita1.xml", "label": "Italian (Anonymous, 1835)", "class": "italian-text"},
            "1st1k-lat2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-lat2.xml", "label": "Latin (Fr. Haase, 1869)", "class": "latin-text"}
        }
    },
    "tlg0086.tlg034": {
        "textgroup": "tlg0086",
        "work": "tlg034",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml", "label": "Greek (Kassel, 1965)", "class": "greek-text"},
             "moerbeke1277-lat1": {"path": "/Users/gcrane/github/Poetics2.0/data/tlg0086/tlg034/tlg0086.tlg034.moerbeke1277-lat1.xml", "label": "Latin (Moerbeke, 1277)", "class": "latin-text"},
           "digicorpus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml", "label": "Greek (Digital Corpus Variant)", "class": "greek-text"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml", "label": "English (W.H. Fyfe, 1927)", "class": "english-text"},
            "butcher1911-eng2": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml", "label": "English (S.H. Butcher, 1911)", "class": "english-text"},
            "bywater1909-eng1": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.bywater1909-eng1.xml", "label": "English (Ingram Bywater, 1909)", "class": "english-text"},
            "twining1789-eng1": {"path": "/Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.twining1789-eng1.xml", "label": "English (Thomas Twining, 1789)", "class": "english-text", "parse_mode": "milestones"}
        }
    },
    "tlg0011.tlg004": {
        "textgroup": "tlg0011",
        "work": "tlg004",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        }
    }
}

def init_storage_engine(db_path):
    if db_path.exists():
        db_path.unlink()
        
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    # Enforce precise 4KB paging limits to streamline local virtual range isolation
    cursor.execute("PRAGMA page_size = 4096;")
    cursor.execute("PRAGMA journal_mode = OFF;")
    cursor.execute("PRAGMA synchronous = OFF;")
    
    cursor.execute("""
        CREATE TABLE text_units (
            canonical_id TEXT PRIMARY KEY,
            urn TEXT NOT NULL,
            label TEXT NOT NULL,
            text_class TEXT NOT NULL,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            short_id TEXT NOT NULL
        );
    """)
    cursor.execute("""
        CREATE TABLE alignment_grid (
            passage_urn TEXT PRIMARY KEY,
            textgroup TEXT NOT NULL,
            work TEXT NOT NULL,
            book TEXT,
            chapter TEXT NOT NULL,
            section TEXT NOT NULL,
            prev_urn TEXT,
            next_urn TEXT,
            sort_order INTEGER NOT NULL
        );
    """)
    cursor.execute("""
        CREATE TABLE text_segments (
            passage_urn TEXT,
            version_short_id TEXT,
            content_html TEXT NOT NULL,
            PRIMARY KEY (passage_urn, version_short_id),
            FOREIGN KEY (passage_urn) REFERENCES alignment_grid(passage_urn)
        );
    """)
    
    cursor.execute("CREATE INDEX idx_grid_lookup ON alignment_grid(textgroup, work, book, chapter);")
    cursor.execute("CREATE INDEX idx_segments_lookup ON text_segments(passage_urn);")
    
    conn.commit()
    return conn

def generate_canonical_id(work_key, label_string):
    prefix = work_key.replace('.', '_')
    match = re.search(r'\((.*?)\)', label_string)
    if match:
        content = match.group(1)
        parts = content.split(',')
        editor = parts[0].strip()
        year = parts[1].strip() if len(parts) > 1 else ""
        editor_last = editor.split()[-1].lower().replace('.', '')
        year_clean = re.sub(r'\D', '', year)
        return f"{prefix}_{editor_last}_{year_clean}"
    fallback_suffix = re.sub(r'\W+', '_', label_string).lower()
    return f"{prefix}_{fallback_suffix}"

def find_text_root(root):
    for div in root.findall('.//{http://www.tei-c.org/ns/1.0}div') + root.findall('.//div'):
        if div.get('type') == 'translation': return div
    return root.find('.//{http://www.tei-c.org/ns/1.0}body') or root.find('.//body') or root.find('.//*body')

def extract_text_recursive(elem, strip_paragraphs=False):
    parts = []
    tag = elem.tag.split('}')[-1]
    
    if tag == 'l':
        line_num = (elem.get('n') or '').strip()
        # Create a line number cell if a line number exists
        if line_num:
            parts.append(f'<div class="line-num-cell">{line_num}</div>')
        else:
            parts.append('<div class="line-num-cell">&nbsp;</div>')
        # Open the matching content text cell
        parts.append('<div class="line-text-cell">')
    elif tag == 'speaker': 
        parts.append('<strong class=\"speaker-attr\">')
    elif tag == 'stage':
        parts.append('<div class=\"stage-direction\">')
    elif tag == 'hi':
        rend = elem.get('rend', 'italic')
        parts.append(f'<span class=\"render-{rend}\">')
    elif tag == 'quote':
        q_type = elem.get('type', 'blockquote')
        parts.append(f'<blockquote class=\"quote-block type-{q_type}\">')
    elif tag == 'p' and not strip_paragraphs:
        parts.append('<p>')

    if elem.text: parts.append(elem.text)
    for child in elem:
        child_tag = child.tag.split('}')[-1]
        if child_tag == 'note':
            note_text = extract_text_recursive(child, strip_paragraphs).strip()
            if note_text: parts.append(f'<span class=\"note\">[{note_text}]</span>')
        elif child_tag == 'lb': parts.append('<br/>')
        else: parts.append(extract_text_recursive(child, strip_paragraphs))
        if child.tail: parts.append(child.tail)
            
    if tag == 'l': 
        parts.append('</div>') # Closes <div class="line-text-cell">
    elif tag == 'speaker': parts.append(': </strong>')
    elif tag == 'stage': parts.append('</div>')
    elif tag == 'hi': parts.append('</span>')
    elif tag == 'quote': parts.append('</blockquote>')
    elif tag == 'p' and not strip_paragraphs: parts.append('</p>')
    return ''.join(parts)

def parse_hierarchical_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    text_entry = find_text_root(tree.getroot())
    if text_entry is None: return None
    data = OrderedDict()

    def walk_divisions(node, current_path):
        tag = node.tag.split('}')[-1]
        subtype = node.get('subtype') or node.get('type')
        n_val = node.get('n')

        if tag == 'div' and subtype in ('book', 'chapter', 'section', 'part', 'textpart') and n_val:
            resolved_subtype = 'chapter' if subtype in ('part', 'textpart') and n_val.isdigit() else subtype
            new_path = current_path + [(resolved_subtype, str(n_val).strip())]
        else:
            new_path = current_path

        paragraphs = node.findall('{http://www.tei-c.org/ns/1.0}p') or node.findall('p')
        if paragraphs and new_path:
            key_map = {t: v for t, v in new_path}
            bk = key_map.get('book', '1')
            ch = key_map.get('chapter', key_map.get('part', '1'))
            sec = key_map.get('section', n_val or '1')

            if bk not in data: data[bk] = OrderedDict()
            if ch not in data[bk]: data[bk][ch] = OrderedDict()
            
            combined_txt = ' '.join(extract_text_recursive(p, strip_paragraphs=False).strip() for p in paragraphs)
            if combined_txt: data[bk][ch][sec] = combined_txt
            return

        for child in node: walk_divisions(child, new_path)

    walk_divisions(text_entry, [])
    return data

def parse_milestone_aligned_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    text_entry = find_text_root(tree.getroot())
    if text_entry is None: return None

    data = OrderedDict()
    current_ch = "1"
    current_sec = "1"
    current_text = []

    def _flush():
        nonlocal current_ch, current_sec, current_text
        if current_text:
            bk = '1'
            data.setdefault(bk, OrderedDict()).setdefault(current_ch, OrderedDict())
            html_blob = ' '.join(t.strip() for t in current_text if t.strip())
            if html_blob:
                if current_sec in data[bk][current_ch]:
                    data[bk][current_ch][current_sec] += " " + html_blob
                else:
                    data[bk][current_ch][current_sec] = html_blob
        current_text = []

    def _walk(elem):
        nonlocal current_ch, current_sec, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone':
            unit = elem.get('unit') or ''
            val = (elem.get('n') or '').strip()
            if val:
                if unit == 'bekker' and '.' in val:
                    parts = val.split('.', 1)
                    _flush()
                    current_ch = parts[0]
                    current_sec = parts[1]
                else:
                    num_match = re.match(r'^\d+', val)
                    if num_match:
                        if unit in ('chapter', 'chapter-start', ''):
                            _flush()
                            current_ch = str(int(num_match.group(0)))
                            current_sec = "1"
                        elif unit in ('section', 'section-start'):
                            _flush()
                            current_sec = str(int(num_match.group(0)))
                        
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag == 'note':
            note_html = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag in ('hi', 'foreign', 'emph'):
            rend = elem.get('rend', 'italic')
            current_text.append(f'<span class="render-{rend}">')
            if elem.text: current_text.append(elem.text)
            for child in elem: _walk(child)
            current_text.append('</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag in ('p', 'div'):
            if elem.text and elem.text.strip(): current_text.append(elem.text)
            for child in elem: _walk(child)
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()  
    return data

def build_poetry_canonical_intervals(editions_dict):
    grc_cfg = editions_dict.get('perseus-grc2') or list(editions_dict.values())[0]
    tree = ET.parse(grc_cfg["path"])
    text_entry = find_text_root(tree.getroot())
    
    landmarks = []
    for elem in text_entry.iter():
        if elem.tag.endswith('milestone') and elem.get('unit') == 'card':
            landmarks.append(('card', elem.get('n').strip()))
        elif elem.tag.endswith('l'):
            ln = (elem.get('n') or '').strip()
            if ln: landmarks.append(('line', ln))

    intervals = []
    for idx, item in enumerate(landmarks):
        if item[0] == 'card':
            card_n = item[1]
            first_l, last_l = None, None
            for ahead in landmarks[idx+1:]:
                if ahead[0] == 'card': break
                if ahead[0] == 'line':
                    if first_l is None: first_l = ahead[1]
                    last_l = ahead[1]
            if not first_l: first_l = card_n
            if not last_l: last_l = first_l
            intervals.append({'card_n': card_n, 'label': f"{first_l}-{last_l}"})
    return intervals

def parse_poetry_cards_tei(path, master_intervals):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    text_entry = find_text_root(tree.getroot())

    data = OrderedDict()
    data["1"] = OrderedDict()
    
    current_label = master_intervals[0]['label'] if master_intervals else "1"
    current_text = []

    def _flush():
        nonlocal current_label, current_text
        if current_text and current_label:
            html = ' '.join(t.strip() for t in current_text if t.strip())
            if html:
                if current_label not in data["1"]:
                    data["1"][current_label] = OrderedDict()
                data["1"][current_label]["1"] = html
        current_text = []

    def _walk(elem):
        nonlocal current_label, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone' and elem.get('unit') == 'card':
            val = (elem.get('n') or '').strip()
            match = [r for r in master_intervals if r['card_n'] == val]
            if match:
                _flush()
                current_label = match[0]['label']
        elif tag in ('l', 'stage'):
            current_text.append(extract_text_recursive(elem, strip_paragraphs=True))
            return
        elif tag == 'note':
            note_html = extract_text_recursive(elem, strip_paragraphs=True).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()
    return data

# ── Processing & Database Compilations ────────────────────────
conn = init_storage_engine(DB_PATH)
cursor = conn.cursor()
global_sort_index = 0

for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    editions = work_meta["editions"]
    
    print(f"Ingesting structural alignment mappings for work: {work_key}...")
    
    for v_id, cfg in editions.items():
        canonical_id = generate_canonical_id(work_key, cfg["label"])
        cursor.execute("""
            INSERT OR REPLACE INTO text_units (canonical_id, urn, label, text_class, textgroup, work, short_id)
            VALUES (?, ?, ?, ?, ?, ?, ?);
        """, (canonical_id, f"urn:cts:greekLit:{work_key}.{v_id}", cfg["label"], cfg["class"], tg, wk, v_id))

    master_intervals = None
    is_poetry = any(cfg.get("parse_mode") == "poetry_cards" for cfg in editions.values())
    if is_poetry:
        master_intervals = build_poetry_canonical_intervals(editions)

    work_corpus = OrderedDict()
    for v_id, cfg in editions.items():
        p_mode = cfg.get("parse_mode")
        if p_mode == "poetry_cards":
            parsed = parse_poetry_cards_tei(cfg["path"], master_intervals)
        elif p_mode == "milestones":
            parsed = parse_milestone_aligned_tei(cfg["path"])
        else:
            parsed = parse_hierarchical_tei(cfg["path"])
            
        if parsed is not None and sum(len(secs) for chs in parsed.values() for secs in chs.values()) > 0:
            work_corpus[v_id] = parsed
            print(f"  ✓ {v_id}: {sum(len(secs) for chs in parsed.values() for secs in chs.values())} segments parsed")
        else:
            print(f"  ✗ {v_id}: Failed to parse completely.")

    if not work_corpus: continue

    first_version = list(work_corpus.keys())[0]
    baseline_corpus = work_corpus[first_version]
    has_multiple_books = len(baseline_corpus.keys()) > 1

    chapter_sequence = []
    for b_k, ch_v in baseline_corpus.items():
        for c_k in ch_v.keys():
            chapter_sequence.append({'book': b_k if has_multiple_books else None, 'chapter': c_k})

    for c_idx, coord in enumerate(chapter_sequence):
        bk_id = coord['book']
        ch_id = coord['chapter']
        lookup_bk = bk_id if bk_id else list(baseline_corpus.keys())[0]
        baseline_secs = list(baseline_corpus[lookup_bk][ch_id].keys())
        
        for sec in baseline_secs:
            global_sort_index += 1
            if bk_id:
                passage_urn = f"urn:cts:greekLit:{work_key}:{bk_id}.{ch_id}.{sec}"
                prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['book']}.{chapter_sequence[c_idx-1]['chapter']}.{sec}" if c_idx > 0 else None
                next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['book']}.{chapter_sequence[c_idx+1]['chapter']}.{sec}" if c_idx < len(chapter_sequence) - 1 else None
            else:
                passage_urn = f"urn:cts:greekLit:{work_key}:{ch_id}.{sec}"
                prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['chapter']}.{sec}" if c_idx > 0 else None
                next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['chapter']}.{sec}" if c_idx < len(chapter_sequence) - 1 else None
            
            cursor.execute("""
                INSERT OR REPLACE INTO alignment_grid 
                (passage_urn, textgroup, work, book, chapter, section, prev_urn, next_urn, sort_order)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
            """, (passage_urn, tg, wk, bk_id, ch_id, sec, prev_urn, next_urn, global_sort_index))
            
            for v_id in editions:
                ch_data = work_corpus.get(v_id, {}).get(lookup_bk, {}).get(ch_id, {})
                raw_html = ch_data.get(sec, "[Text range missing in alignment layer]")
                cursor.execute("""
                    INSERT OR REPLACE INTO text_segments (passage_urn, version_short_id, content_html)
                    VALUES (?, ?, ?);
                """, (passage_urn, v_id, raw_html))

# Commit transaction structures cleanly ahead of disk reorganizations
conn.commit()

# Run the structural page defragmenter to clean the allocation layer
cursor.execute("VACUUM;")
conn.close()

print(f"\n[SUCCESS] Serverless Relational Storage Asset compiled cleanly at: {DB_PATH}")

Ingesting structural alignment mappings for work: tlg0003.tlg001...
  ✓ perseus-grc2: 3587 segments parsed
  ✓ 1st1K-eng1: 3580 segments parsed
  ✓ 1st1K-eng2: 3579 segments parsed
  ✓ perseus-eng4: 3582 segments parsed
  ✓ perseus-eng6: 3587 segments parsed
  ✓ 1st1k-fre1: 917 segments parsed
  ✓ 1st1K-fre2: 917 segments parsed
  ✓ 1st1K-ger1: 59 segments parsed
  ✓ 1st1K-ger2: 930 segments parsed
  ✓ 1st1K-ger3: 923 segments parsed
  ✓ 1st1K-ger4: 50 segments parsed
  ✓ 1st1K-ita1: 917 segments parsed
  ✓ 1st1k-lat2: 3624 segments parsed
Ingesting structural alignment mappings for work: tlg0086.tlg034...
  ✓ perseus-grc2: 381 segments parsed
  ✓ moerbeke1277-lat1: 368 segments parsed
  ✓ digicorpus-grc2: 381 segments parsed
  ✓ perseus-eng2: 381 segments parsed
  ✓ butcher1911-eng2: 381 segments parsed
  ✓ bywater1909-eng1: 380 segments parsed
  ✓ twining1789-eng1: 380 segments parsed
Ingesting structural alignment mappings for work: tlg0011.tlg004...
  ✓ perseus-grc2: 72 segments pa

In [3]:
import json
import sqlite3
from pathlib import Path

# ── Target Configuration ───────────────────────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/github/persverscomp")
DB_PATH = WORKSPACE_DIR / "corpus_alignment_grid.db"

# ── Dynamic Relational Extraction for Frontend State ───────────
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database asset not found at {DB_PATH}. Please execute Cell 1 first.")

conn = sqlite3.connect(str(DB_PATH))
cursor = conn.cursor()

# 1. Dynamically reconstruct GLOBAL_REGISTRIES from text_units table
cursor.execute("SELECT canonical_id, urn, label, text_class, textgroup, work, short_id FROM text_units")
extracted_registries = {}
for row in cursor.fetchall():
    extracted_registries[row[0]] = {
        "urn": row[1],
        "label": row[2],
        "class": row[3],
        "textgroup": row[4],
        "work": row[5],
        "short_id": row[6]
    }

# 2. Dynamically reconstruct GLOBAL_STRUCTURES from alignment_grid table
extracted_structures = {}
cursor.execute("SELECT DISTINCT textgroup || '.' || work FROM alignment_grid")
work_keys = [r[0] for r in cursor.fetchall()]

for w_key in work_keys:
    tg, wk = w_key.split('.')
    
    # Check if this work uses multi-book navigation or a flat structure
    cursor.execute("SELECT DISTINCT book FROM alignment_grid WHERE textgroup=? AND work=? AND book IS NOT NULL", (tg, wk))
    books = [r[0] for r in cursor.fetchall()]
    
    if books:
        # Multi-book configuration (e.g., Thucydides)
        book_map = {}
        for bk in sorted(books, key=lambda x: int(x) if x.isdigit() else x):
            cursor.execute("SELECT DISTINCT chapter FROM alignment_grid WHERE textgroup=? AND work=? AND book=? ORDER BY sort_order", (tg, wk, bk))
            book_map[bk] = [r[0] for r in cursor.fetchall()]
        extracted_structures[w_key] = book_map
    else:
        # Flat configuration (e.g., Aristotle, Sophocles)
        cursor.execute("SELECT DISTINCT chapter FROM alignment_grid WHERE textgroup=? AND work=? ORDER BY sort_order", (tg, wk))
        extracted_structures[w_key] = [r[0] for r in cursor.fetchall()]

conn.close()

# Serialize configurations directly into JS injection tokens
struct_map_json = json.dumps(extracted_structures)
text_registry_json = json.dumps(extracted_registries)


# ── 1. Compile Repository README.md Asset ─────────────────────
README_MD_CONTENT = """# Minimum Viable Perseus (MVP) | Project Parallel Workspace

## Infrastructure Sustainability Paradigm
Rather than relying on continuous, monolithic server daemons or generating hundreds of thousands of fragmented, static JavaScript segment chunks over a fragile local filesystem, true architectural sustainability demands a **decentralized serverless delivery network**. 

By compiling multi-text alignment grids directly into single, self-contained binary data assets using highly stable open formats like SQLite, we achieve an archive designed for permanent digital preservation. When hosted on passive, globally distributed object networks, the browser leverages asynchronous WebAssembly to download only local, isolated byte ranges dynamically via HTTP Range Requests. This eliminates the operational footprint, infrastructure fragility, and ongoing maintenance costs of legacy backend servers—providing a robust platform for permanent digital philology that scales gracefully to the full size of the human record.
"""
(WORKSPACE_DIR / "README.md").write_text(README_MD_CONTENT, encoding='utf-8')


# ── 2. Compile Branded HTML Workspace Application ──────────────
INDEX_HTML_CONTENT = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Minimum Viable Perseus (MVP) | Dual-Framework Workspace</title>
    <script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
    
    <script src="https://cdnjs.cloudflare.com/ajax/libs/sql.js/1.8.0/sql-wasm.js"></script>
    
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        html, body {
            height: 100%; width: 100%;
            font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
            font-size: 14px; color: #111; background: #fdfdfa;
        }
        a { color: #880000; text-decoration: none; font-weight: bold; }
        a:hover { text-decoration: underline; }

        /* Splash Page Branding Layout */
        #splash-view-root { display: flex; flex-direction: column; width: 100%; min-height: 100vh; justify-content: center; align-items: center; background: #fdfdfa; padding: 20px; }
        #splash-wrapper { max-width: 720px; width: 100%; text-align: center; padding: 40px; border: 2px dashed #660000; background: #fff; border-radius: 6px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); }
        
        .brand-banner-frame { display: flex; justify-content: center; align-items: center; border-bottom: 2px solid #660000; padding-bottom: 20px; margin-bottom: 20px; }
        .brand-banner-frame img { max-width: 100%; height: auto; display: block; }

        .file-upload-zone { margin: 25px 0; padding: 25px 20px; border: 2px dashed #ccc; background: #fafafa; border-radius: 4px; cursor: pointer; position: relative; transition: all 0.2s ease; }
        .file-upload-zone:hover { border-color: #660000; background: #fdfbef; }
        .file-input-hidden { position: absolute; top: 0; left: 0; width: 100%; height: 100%; opacity: 0; cursor: pointer; }

        .workspace-emblem-holder { display: flex; align-items: center; justify-content: center; }
        .workspace-emblem-holder img { height: 32px; width: auto; display: block; border-radius: 2px; }

        /* Dedicated Launcher Framework Element */
        .launch-control-container { margin: 25px 0; padding: 20px; background: #fdfbef; border: 1px solid #e6d3a8; border-radius: 4px; display: none; text-align: center; }
        .launch-btn-primary { display: inline-block; background: #660000; color: #fff !important; font-family: Georgia, serif; font-size: 16px; font-weight: bold; padding: 12px 30px; border: none; border-radius: 4px; cursor: pointer; box-shadow: 0 2px 5px rgba(0,0,0,0.15); transition: background 0.15s; }
        .launch-btn-primary:hover { background: #330000; text-decoration: none; }

        /* Dynamic Markdown Output Styling */
        #manifesto-render-target h1 { display: none; } 
        #manifesto-render-target h2 { font-size: 18px; color: #333; margin-top: 28px; margin-bottom: 12px; font-variant: small-caps; border-bottom: 1px dashed #e2e2d5; padding-bottom: 5px; text-align: left; }
        #manifesto-render-target p { line-height: 1.6; margin-bottom: 14px; color: #333; text-align: justify; }
        #manifesto-render-target blockquote { border-left: 3px solid #660000; padding: 10px 0 10px 15px; margin: 20px 0; font-style: italic; color: #444; background: #fefefe; }
        #manifesto-render-target ul { margin-left: 25px; margin-bottom: 20px; line-height: 1.6; text-align: left; }
        #manifesto-render-target li { margin-bottom: 8px; color: #333; }
        #manifesto-render-target strong { color: #000; }

        /* Active Workspace Navigation Layout */
        #app-view-root { display: none; flex-direction: column; height: 100vh; width: 100vw; overflow: hidden; background: #fff; }
        #header-container { flex-shrink: 0; background: #fff; border-bottom: 1px solid #ccc; z-index: 10; }
        
        #perseus-banner { background: #660000; color: #fff; padding: 5px 15px; display: flex; justify-content: space-between; align-items: center; }
        .banner-logo-title-group { display: flex; align-items: center; gap: 12px; }
        .banner-logo-title-group h1 a { color: #fff; font-size: 16px; font-weight: bold; font-family: Georgia, serif; }

        .mode-toggle-panel { display: flex; gap: 8px; align-items: center; }
        .toggle-btn { background: #440000; color: #ffccaa; border: 1px solid #993333; font-size: 11px; padding: 3px 10px; border-radius: 3px; cursor: pointer; font-weight: bold; }
        .toggle-btn.active-mode { background: #fff; color: #660000; border-color: #fff; }

        #nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 4px 15px; font-size: 11px; font-weight: bold; }
        #browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 6px 15px; font-size: 11px; display: flex; flex-wrap: wrap; gap: 5px; }
        .browse-row { display: flex; align-items: flex-start; width: 100%; }
        .browse-row.hidden-row { display: none !important; }
        .browse-row label { font-weight: bold; width: 80px; color: #444; flex-shrink: 0; padding-top: 1px; }
        .browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
        .browse-items a { padding: 1px 6px; background: #e0e0d0; color: #336699; border: 1px solid #bbbb99; font-weight: bold; border-radius: 2px; cursor: pointer; }
        .browse-items a.current { background: #660000; color: #fff; border-color: #330000; }

        .browse-items a.sec-pill { background: #fff; color: #555; border: 1px solid #ccc; font-weight: normal; font-size: 10px; padding: 1px 5px; }
        .browse-items a.sec-pill.active-pill { background: #660000; color: #fff; border-color: #330000; font-weight: bold; }

        #outer-wrapper { display: flex; width: 100%; flex: 1; min-height: 0; overflow: hidden; }

        /* View Mode Layout Grid Mapping Rules */
        #main-container { display: flex; width: 100%; height: 100%; min-width: 0; position: relative; }
        .reading-column { display: flex; flex-direction: column; border-right: 1px solid #eeeeee; background: #fff; min-width: 0; height: 100%; }
        .reading-column:last-child { border-right: none; }
        .cross-panel { background: #fafafa; }
        .column-content-scroll { flex: 1; overflow-y: auto; padding: 15px; }
        .panel-header { background: #660000; color: #fff; padding: 4px 8px; font-size: 11px; font-weight: bold; position: sticky; top: 0; z-index: 5; display: flex; justify-content: space-between; align-items: center; flex-shrink: 0; }
        .edition-select-dropdown { background: #fff; color: #333; font-size: 10px; padding: 1px 4px; border: 1px solid #ccc; border-radius: 2px; outline: none; cursor: pointer; }

        /* Left Hand Rail Navigation */
        #classic-left-rail { display: none; width: auto; min-width: 80px; max-width: 320px; flex-shrink: 0; background: #f4f4f0; border-right: 1px solid #dcdcc8; padding: 10px 12px; overflow-y: auto; }
        #classic-left-rail h3 { font-size: 12px; font-variant: small-caps; color: #555; margin-bottom: 8px; border-bottom: 1px solid #ccccbb; padding-bottom: 3px; }
        .rail-tree-item { display: block; font-size: 12px; padding: 3px 6px; color: #336699; font-weight: bold; border-radius: 2px; margin-bottom: 2px; cursor: pointer; white-space: nowrap; }
        .rail-tree-item:hover { background: #e8e8df; }
        .rail-tree-item.active-rail-node { background: #660000; color: #fff !important; }

        /* Layout Types Rules */
        #outer-wrapper.mode-parallel #main-container { flex-direction: row; }
        #outer-wrapper.mode-parallel .reading-column { flex: 1; height: 100%; }
        #outer-wrapper.mode-parallel #classic-left-rail { display: none; }

        #outer-wrapper.mode-classic #classic-left-rail { display: block; }
        #outer-wrapper.mode-classic #main-container { 
            display: grid; 
            grid-template-columns: 3fr 2fr; 
            grid-template-rows: 1fr 1fr; 
            height: 100%; width: 100%;
        }
        #outer-wrapper.mode-classic #col_f { grid-column: 1; grid-row: 1 / span 2; border-right: 2px solid #ccc; height: 100%; min-height: 0; }
        #outer-wrapper.mode-classic #col_c1 { grid-column: 2; grid-row: 1; height: 100%; min-height: 0; border-bottom: 1px solid #ccc; }
        #outer-wrapper.mode-classic #col_c2 { grid-column: 2; grid-row: 2; height: 100%; min-height: 0; border-top: 1px solid #ccc; }

        /* Typographic Content Rendering Elements */
        .section-row { border-bottom: 1px solid #f0f0f0; padding: 10px 0; display: block; scroll-margin-top: 5px; }
        .section-row.hidden-section { display: none !important; }
        .greek-text { font-size: 15px; line-height: 1.7; font-family: "Gentium Plus", "Athena", serif; color: #000; }
        .english-text { font-size: 13px; line-height: 1.6; color: #222; }
        .french-text { font-size: 13px; line-height: 1.6; color: #2b2b2b; }
        .latin-text { font-size: 13px; line-height: 1.6; color: #111; font-style: italic; }
        .german-text, .italian-text { font-size: 13px; line-height: 1.6; color: #222; }

        .poetry-grid-layout { display: grid !important; grid-template-columns: 45px 1fr; row-gap: 4px; align-items: start; width: 100%; }
        .poetry-grid-layout .line-num-cell { grid-column: 1; text-align: right; padding-right: 12px; font-size: 11px; color: #999; font-weight: bold; font-family: monospace; line-height: 1.7; user-select: none; display: block !important; }
        .poetry-grid-layout .line-text-cell { grid-column: 2; line-height: 1.7; display: block !important; white-space: normal; }
        .poetry-grid-layout .stage-direction { grid-column: 1 / span 2 !important; display: block !important; font-style: italic; color: #445566; background: #fdfdfd; border-left: 3px solid #ccc; padding: 8px 12px; margin: 10px 0; line-height: 1.6; font-size: 0.95em; }

        .prose-inline-layout { display: block; width: 100%; line-height: 1.6; }
        .prose-inline-layout .prose-marker { font-weight: bold; color: #336699; margin-right: 8px; display: inline-block; }
        .prose-inline-layout .prose-body-inline { display: inline; }
        .prose-body-inline p, .prose-body-inline div { display: inline !important; margin: 0 !important; padding: 0 !important; }

        .speaker-attr { color: #445566; font-variant: small-caps; letter-spacing: 0.5px; display: block; margin-top: 8px; margin-bottom: 4px; }
        .poetry-grid-layout .speaker-attr { grid-column: 1 / span 2; display: block !important; }
        .render-italic, .render-italics { font-style: italic; }
        .render-bold { font-weight: bold; }
        
        .note { font-size: 11px; color: #555; background: #fdfbf7; border-left: 2px solid #880000; padding: 2px 6px; margin: 4px 0 4px 1em; display: block; }
        .poetry-grid-layout .note { grid-column: 1 / span 2; display: block !important; }

        .quote-block { margin: 10px 0 10px 2.5em; padding-left: 12px; border-left: 1px dashed #bbb; font-size: 0.95em; color: #333; grid-column: 1 / span 2; }
        .quote-block.type-verse { font-style: italic; line-height: 1.6; background: #fafafa; padding: 6px 12px; }
        .quote-block.type-blockquote { line-height: 1.55; color: #1a1a1a; }
        .viewport-footer-controls { display: flex; justify-content: space-between; align-items: center; gap: 8px; margin-top: 20px; padding-top: 10px; border-top: 1px solid #ddd; width: 100%; clear: both; }
        .poetry-grid-layout .viewport-footer-controls { grid-column: 1 / span 2; display: flex !important; }
        .footer-group-left { display: flex; gap: 6px; }
        .action-btn { display: inline-block; background: #660000; color: #fff !important; padding: 4px 10px; font-size: 11px; font-weight: bold; border-radius: 2px; border: none; cursor: pointer; text-align: center; }
        .action-btn.secondary { background: #666; }
        .action-btn:hover { background: #330000; text-decoration: none; }
        .action-btn.secondary:hover { background: #444; }
    
        /* Panel Resizing Mechanics */
        .resize-handle { flex-shrink: 0; background: #e0e0e0; position: relative; z-index: 20; transition: background 0.15s; }
        .resize-handle:hover, .resize-handle.dragging { background: #660000; }
        .resize-handle-h { width: 5px; cursor: col-resize; height: 100%; }
        .resize-handle-v { height: 5px; cursor: row-resize; width: 100%; }
        body.resizing-cols, body.resizing-cols * { cursor: col-resize !important; }
        body.resizing-rows, body.resizing-rows * { cursor: row-resize !important; }

        .make-focus-btn { font-size: 10px; padding: 1px 6px; margin-left: 6px; background: rgba(255,255,255,0.2); color: #fff; border: 1px solid rgba(255,255,255,0.4); border-radius: 3px; cursor: pointer; font-weight: normal; }
        .make-focus-btn:hover { background: rgba(255,255,255,0.35); }

        .toggle-btn.col-scaler-btn { background: #333; color: #aaa; border-color: #555; font-size: 10px; padding: 2px 8px; margin-left: 4px; }

        /* Flexible Multi-Column State Handling Rules */
        #col_c3, #col_c4, #resize-c2-c3, #resize-c3-c4 { display: none; }
        
        .four-columns #col_c3 { display: flex !important; }
        .four-columns #resize-c2-c3 { display: block !important; }

        .five-columns #col_c3, .five-columns #col_c4 { display: flex !important; }
        .five-columns #resize-c2-c3, .five-columns #resize-c3-c4 { display: block !important; }

        .four-columns.mode-parallel .reading-column, \
        .five-columns.mode-parallel .reading-column { flex: 1 !important; width: auto !important; }

        #outer-wrapper.mode-classic.four-columns #main-container { grid-template-columns: 3fr 2fr; grid-template-rows: 1fr 1fr 1fr; }
        #outer-wrapper.mode-classic.four-columns #col_c3 { display: flex !important; border-top: 1px solid #ccc; }

        #outer-wrapper.mode-classic.five-columns #main-container { grid-template-columns: 3fr 2fr; grid-template-rows: 1fr 1fr 1fr 1fr; }
        #outer-wrapper.mode-classic.five-columns #col_c3, \
        #outer-wrapper.mode-classic.five-columns #col_c4 { display: flex !important; border-top: 1px solid #ccc; }
    </style>
</head>
<body>

  <div id="splash-view-root">
      <div id="splash-wrapper">
          <div class="brand-banner-frame">
              <img src="./assets/perseus_banner.png" alt="Perseus Digital Library Banner" />
          </div>

          <div class="launch-control-container" id="launcher-box">
              <button class="launch-btn-primary" onclick="launchWorkspaceDirectly()">Launch Workspace Viewer</button>
              <div style="font-size: 11px; color: #666; margin-top: 8px; font-style: italic;">SQLite database ready. Click above to initialize the reading workspace grid.</div>
          </div>

          <div class="file-upload-zone" id="drop-zone">
              <strong style="color: #660000; font-family: Georgia, serif; font-size: 16px;">Mount Local Data Asset Override</strong><br/>
              <span style="font-size: 12px; color: #555; display: inline-block; margin-top: 8px;">Drag & Drop `corpus_alignment_grid.db` here to replace the hosted database</span>
              <input type="file" id="file-picker" class="file-input-hidden" accept=".db" onchange="handleFileSelection(this.files)"/>
          </div>
          
          <div id="status-readout" style="font-size: 11px; color: #777; font-style: italic; margin-bottom: 15px;">Awaiting compilation asset attachment...</div>
          <div id="manifesto-render-target"></div>
      </div>
  </div>

  <div id="app-view-root">
    <div id="header-container">
      <div id="perseus-banner">
        <div class="banner-logo-title-group">
            <div class="workspace-emblem-holder">
                <img src="./assets/perseus_running_man.png" alt="Perseus Emblem" />
            </div>
            <h1><a href="#" onclick="window.location.reload()">Perseus Workspace Engine</a></h1>
        </div>
        
        <div class="mode-toggle-panel">
            <button id="btn-mode-parallel" class="toggle-btn active-mode" onclick="setWorkspaceLayoutMode('parallel')">Parallel Grid</button>
            <button id="btn-mode-classic" class="toggle-btn" onclick="setWorkspaceLayoutMode('classic')">Classic Asymmetric</button>
            <button id="btn-column-scaler" class="toggle-btn col-scaler-btn" onclick="cycleColumns()">Columns: 3</button>
        </div>
      </div>
      <div id="nav-bar">
        <span id="frame-context-label">Mounting layout frameworks...</span>
      </div>
      
      <div id="browse-bar">
        <div class="browse-row" id="row-work-selector">
          <label>Works:</label>
          <div class="browse-items" id="work-items-container"></div>
        </div>
        <div class="browse-row" id="row-book-container">
          <label>Books:</label>
          <div class="browse-items" id="book-items-container"></div>
        </div>
        <div class="browse-row" id="row-chapter-selector">
          <label id="chapter-row-label">Chapters:</label>
          <div class="browse-items" id="chapter-items-container"></div>
        </div>
        <div class="browse-row" id="row-section-selector">
          <label>Sections:</label>
          <div class="browse-items" id="section-items-container"></div>
        </div>
      </div>
    </div>

    <div id="outer-wrapper" class="mode-parallel">
      <div id="classic-left-rail">
          <h3 id="rail-title">Structure Rail</h3>
          <div id="rail-nodes-target"></div>
      </div>

      <div id="main-container">
        <div class="reading-column" id="col_f">
            <div class="panel-header">
                <span>Focus</span>
                <select class="edition-select-dropdown" id="select_f" onchange="updateColumnContent('f', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_f"></div>
        </div>
        <div class="resize-handle resize-handle-h" id="resize-f-c1"></div>
        <div class="reading-column cross-panel" id="col_c1">
            <div class="panel-header">
                <span>Comparison Version 1 <button class="make-focus-btn" onclick="swapToFocus('c1')">⇦ Focus</button></span>
                <select class="edition-select-dropdown" id="select_c1" onchange="updateColumnContent('c1', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c1"></div>
        </div>
        <div class="resize-handle resize-handle-h" id="resize-c1-c2"></div>
        <div class="reading-column cross-panel" id="col_c2">
            <div class="panel-header">
                <span>Comparison Version 2 <button class="make-focus-btn" onclick="swapToFocus('c2')">⇦ Focus</button></span>
                <select class="edition-select-dropdown" id="select_c2" onchange="updateColumnContent('c2', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c2"></div>
        </div>
        <div class="resize-handle resize-handle-h" id="resize-c2-c3"></div>
        <div class="reading-column cross-panel" id="col_c3">
            <div class="panel-header">
                <span>Comparison Version 3 <button class="make-focus-btn" onclick="swapToFocus('c3')">⇦ Focus</button></span>
                <select class="edition-select-dropdown" id="select_c3" onchange="updateColumnContent('c3', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c3"></div>
        </div>
        <div class="resize-handle resize-handle-h" id="resize-c3-c4"></div>
        <div class="reading-column cross-panel" id="col_c4">
            <div class="panel-header">
                <span>Comparison Version 4 <button class="make-focus-btn" onclick="swapToFocus('c4')">⇦ Focus</button></span>
                <select class="edition-select-dropdown" id="select_c4" onchange="updateColumnContent('c4', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c4"></div>
        </div>
      </div>
    </div>
  </div>

  <script>
    const GLOBAL_STRUCTURES = STRUCT_REPLACE;
    const TEXT_REGISTRY = REGISTRY_REPLACE;
    
    let dbInstance = null;
    let SQL_WASM_ENGINE = null;
    
    let activeWorkKey = "tlg0003.tlg001";
    let activeUrnContext = "";
    let activeSectionFilter = null;
    let currentActiveMode = "parallel";
    let activeColumnsCount = 3;
    let columnEditions = { f: "", c1: "", c2: "", c3: "", c4: "" };

    // Fetch repository manifesto text asset dynamically to render home page documentation
    fetch('./README.md')
        .then(res => res.text())
        .then(text => {
            if (window.marked && window.marked.parse) {
                document.getElementById('manifesto-render-target').innerHTML = window.marked.parse(text);
            }
        }).catch(err => console.warn("README fallback skipped:", err));

    initSqlJs({ locateFile: file => `https://cdnjs.cloudflare.com/ajax/libs/sql.js/1.8.0/${file}` }).then(SQL => {
        SQL_WASM_ENGINE = SQL;
        document.getElementById("status-readout").innerText = "WebAssembly layer compiled. Accessing database assets...";
        
        fetch('./corpus_alignment_grid.db')
            .then(response => {
                if (!response.ok) throw new Error("Database asset not found on backend distribution.");
                return response.arrayBuffer();
            })
            .then(buffer => {
                const uInt8Array = new Uint8Array(buffer);
                dbInstance = new SQL_WASM_ENGINE.Database(uInt8Array);
                document.getElementById("status-readout").innerText = "Database connection mounted successfully.";
                
                // Inspect search state parameters to determine routing behavior
                const params = new URLSearchParams(window.location.search);
                if (params.has("w")) {
                    // Prepopulated deep link detected: Bypass landing layout completely
                    initializeRoutingFromURL();
                } else {
                    // Bare entry root: reveal the workspace manual button trigger box
                    document.getElementById("launcher-box").style.display = "block";
                }
            })
            .catch(err => {
                console.warn(err.message);
                document.getElementById("status-readout").innerText = "Hosted asset skipped. Awaiting manual grid selection upload...";
            });

    }).catch(err => {
        console.error(err);
        document.getElementById("status-readout").innerText = "WebAssembly core initialization error.";
    });

    window.launchWorkspaceDirectly = function() {
        initializeRoutingFromURL();
    };

    function handleFileSelection(files) {
        if (!files || files.length === 0 || !SQL_WASM_ENGINE) return;
        document.getElementById("status-readout").innerText = "Mounting local parallel frameworks...";
        
        const file = files[0];
        const reader = new FileReader();
        
        reader.onload = function(e) {
            try {
                const buffer = new Uint8Array(e.target.result);
                dbInstance = new SQL_WASM_ENGINE.Database(buffer);
                initializeRoutingFromURL();
            } catch (err) {
                console.error(err);
                document.getElementById("status-readout").innerText = "Error: Selected format is not a valid alignment database.";
            }
        };
        reader.readAsArrayBuffer(file);
    }

    function isFlatStructure(wKey) { return Array.isArray(GLOBAL_STRUCTURES[wKey]); }
    function isPoetryWork(wKey) { return wKey.startsWith("tlg0011."); }

    function getChapterDataPayload(workKey, book, chapter) {
        const parts = workKey.split(".");
        let queryStr = `SELECT passage_urn, section, prev_urn, next_urn FROM alignment_grid WHERE textgroup='${parts[0]}' AND work='${parts[1]}' AND chapter='${chapter}'`;
        if (book) queryStr += ` AND book='${book}'`;
        queryStr += " ORDER BY sort_order";
        
        const gridResult = dbInstance.exec(queryStr);
        if (gridResult.length === 0) return null;
        
        const baseRows = gridResult[0].values;
        const sectionsPayload = {};
        let masterUrn = "";
        let navigation = { prev: null, next: null };
        
        baseRows.forEach(([pUrn, sec, prevUrn, nextUrn]) => {
            if(!masterUrn) masterUrn = pUrn.substring(0, pUrn.lastIndexOf('.'));
            navigation.prev = prevUrn;
            navigation.next = nextUrn;
            sectionsPayload[sec] = {};
            
            const contentQuery = dbInstance.exec(`SELECT version_short_id, content_html FROM text_segments WHERE passage_urn='${pUrn}'`);
            if (contentQuery.length > 0) {
                contentQuery[0].values.forEach(([vId, html]) => {
                    sectionsPayload[sec][vId] = html;
                });
            }
        });
        
        return {
            urn: masterUrn, textgroup: parts[0], work: parts[1], book: book, chapter: chapter,
            sections: sectionsPayload, navigation: navigation
        };
    }

    function populateDropdownsForWork(tg, wk) {
        const workKey = `${tg}.${wk}`;
        const validEditions = Object.entries(TEXT_REGISTRY).filter(([key, meta]) => {
            return meta.textgroup === tg && meta.work === wk;
        });

        ['f', 'c1', 'c2', 'c3', 'c4'].forEach((prefix, index) => {
            const selectEl = document.getElementById(`select_${prefix}`);
            if (!selectEl) return;
            
            selectEl.innerHTML = "";
            
            validEditions.forEach(([canonicalId, meta]) => {
                const opt = document.createElement("option");
                opt.value = canonicalId;
                opt.innerText = meta.label;
                selectEl.appendChild(opt);
            });

            if (!columnEditions[prefix] && validEditions.length > 0) {
                const editionIndex = index % validEditions.length;
                columnEditions[prefix] = validEditions[editionIndex][0];
            }
            
            selectEl.value = columnEditions[prefix];
        });
    }

    function initializeRoutingFromURL() {
        const params = new URLSearchParams(window.location.search);
        const rawParam = params.get("w") || "tlg0003.tlg001";
        let b = null, ch = "1";
        activeSectionFilter = null;
        
        columnEditions.f = params.get("focus") || "";
        columnEditions.c1 = params.get("right") || "";
        columnEditions.c2 = params.get("right2") || "";
        columnEditions.c3 = params.get("right3") || "";
        columnEditions.c4 = params.get("right4") || "";

        const colParam = params.get("cols");
        if (colParam) {
            activeColumnsCount = parseInt(colParam, 10) || 3;
            setTimeout(() => {
                const ow = document.getElementById('outer-wrapper');
                if (activeColumnsCount === 4) ow.classList.add('four-columns');
                if (activeColumnsCount === 5) ow.classList.add('five-columns');
                document.getElementById('btn-column-scaler').innerText = `Columns: ${activeColumnsCount}`;
            }, 0);
        }

        if (rawParam.includes(":")) {
            const querySegments = rawParam.split(":");
            activeWorkKey = querySegments[0];
            const passageSegments = querySegments[1].split(".");
            
            if (isFlatStructure(activeWorkKey)) {
                b = null; ch = passageSegments[0] || "1";
                if (passageSegments[1]) activeSectionFilter = passageSegments[1];
            } else {
                b = passageSegments[0]; ch = passageSegments[1] || "1";
                if (passageSegments[2]) activeSectionFilter = passageSegments[2];
            }
        } else {
            activeWorkKey = rawParam;
            if (isFlatStructure(activeWorkKey)) {
                b = null; ch = GLOBAL_STRUCTURES[activeWorkKey][0] || "1";
            } else {
                b = Object.keys(GLOBAL_STRUCTURES[activeWorkKey])[0];
                ch = GLOBAL_STRUCTURES[activeWorkKey][b][0] || "1";
            }
        }

        const workPrefix = activeWorkKey.split(".")[1];
        ['f', 'c1', 'c2', 'c3', 'c4'].forEach(prefix => {
            const currentEd = columnEditions[prefix];
            if (currentEd && TEXT_REGISTRY[currentEd] && TEXT_REGISTRY[currentEd].work !== workPrefix) {
                columnEditions[prefix] = ""; 
            }
        });

        // Hide splash layer and reveal workspace context layout
        document.getElementById("splash-view-root").style.display = "none";
        document.getElementById("app-view-root").style.display = "flex";

        triggerTargetNavigation(b, ch);
    }

    function triggerTargetNavigation(book, chapter) {
        const payload = getChapterDataPayload(activeWorkKey, book, chapter);
        if (!payload) return;
        activeUrnContext = payload.urn;
        
        document.getElementById("frame-context-label").innerText = `Active Frame Context URN: ${payload.urn}`;
        
        populateDropdownsForWork(payload.textgroup, payload.work);
        renderNavigationControls(payload);
        renderActiveContentLayers(payload);
        
        updateURLState(payload.book, payload.chapter);
    }      

    function triggerViewRefresh() {
        let currentBk = null, currentCh = "1";
        const currentActiveChEl = document.querySelector("#chapter-items-container a.current");
        if (currentActiveChEl) currentCh = currentActiveChEl.innerText;
        
        if (!isFlatStructure(activeWorkKey)) {
            const currentActiveBkEl = document.querySelector("#book-items-container a.current");
            currentBk = currentActiveBkEl ? currentActiveBkEl.innerText : Object.keys(GLOBAL_STRUCTURES[activeWorkKey])[0];
        }
        triggerTargetNavigation(currentBk, currentCh);
    }

    function updateURLState(book, chapter) {
        const params = new URLSearchParams();
        let passageValue = book ? `${book}.${chapter}` : `${chapter}`;
        if (activeSectionFilter) passageValue += `.${activeSectionFilter}`;

        params.set("w", `${activeWorkKey}:${passageValue}`);

        if (columnEditions.f) params.set("focus", columnEditions.f);
        if (columnEditions.c1) params.set("right", columnEditions.c1);
        if (columnEditions.c2) params.set("right2", columnEditions.c2);
        if (columnEditions.c3) params.set("right3", columnEditions.c3);
        if (columnEditions.c4) params.set("right4", columnEditions.c4);
        params.set("cols", activeColumnsCount.toString());

        window.history.replaceState(null, "", window.location.pathname + "?" + params.toString());
    }

    function selectSectionDirectly(secId) {
        activeSectionFilter = (activeSectionFilter === secId) ? null : secId;
        triggerViewRefresh();
    }

    function renderNavigationControls(payload) {
        const isDramaOrPoetry = isPoetryWork(activeWorkKey);
        
        const rowWork = document.getElementById("row-work-selector");
        const rowBook = document.getElementById("row-book-container");
        const rowChapter = document.getElementById("row-chapter-selector");
        const rowSection = document.getElementById("row-section-selector");

        if (currentActiveMode === 'classic') {
            if (rowWork) rowWork.classList.add("hidden-row");
            if (rowBook) rowBook.classList.add("hidden-row");
            if (rowChapter) rowChapter.classList.add("hidden-row");
            if (rowSection) { if (isDramaOrPoetry) rowSection.classList.add("hidden-row"); else rowSection.classList.remove("hidden-row"); }
        } else {
            if (rowWork) rowWork.classList.remove("hidden-row");
            if (rowChapter) rowChapter.classList.remove("hidden-row");
            if (rowSection) { if (isDramaOrPoetry) rowSection.classList.add("hidden-row"); else rowSection.classList.remove("hidden-row"); }
            if (rowBook) { if (isFlatStructure(activeWorkKey)) rowBook.classList.add("hidden-row"); else rowBook.classList.remove("hidden-row"); }
        }

        const workContainer = document.getElementById("work-items-container");
        if (workContainer) {
            workContainer.innerHTML = "";
            Object.keys(GLOBAL_STRUCTURES).forEach(wKey => {
                const a = document.createElement("a");
                if (wKey === "tlg0003.tlg001") a.innerText = "Thucydides (Histories)";
                else if (wKey === "tlg0086.tlg034") a.innerText = "Aristotle (Poetics)";
                else if (wKey === "tlg0011.tlg004") a.innerText = "Sophocles (Oedipus)";
                
                if(wKey === activeWorkKey) a.className = "current";
                a.onclick = () => {
                    activeSectionFilter = null; 
                    activeWorkKey = wKey;
                    columnEditions = { f: "", c1: "", c2: "", c3: "", c4: "" };
                    
                    if (isFlatStructure(wKey)) {
                        triggerTargetNavigation(null, GLOBAL_STRUCTURES[wKey][0]);
                    } else {
                        const targetBook = Object.keys(GLOBAL_STRUCTURES[wKey])[0];
                        triggerTargetNavigation(targetBook, GLOBAL_STRUCTURES[wKey][targetBook][0]);
                    }
                };
                workContainer.appendChild(a);
            });
        }

        if (currentActiveMode !== 'classic' && !isFlatStructure(activeWorkKey)) {
            const bookContainer = document.getElementById("book-items-container");
            if (bookContainer) {
                bookContainer.innerHTML = "";
                Object.keys(GLOBAL_STRUCTURES[activeWorkKey]).forEach(bk => {
                    const a = document.createElement("a"); a.innerText = bk;
                    if(bk === payload.book) a.className = "current";
                    a.onclick = () => { activeSectionFilter = null; triggerTargetNavigation(bk, GLOBAL_STRUCTURES[activeWorkKey][bk][0]); };
                    bookContainer.appendChild(a);
                });
            }
        }

        const bookLabelEl = document.getElementById("chapter-row-label");
        if (bookLabelEl) bookLabelEl.innerText = isDramaOrPoetry ? "Lines:" : "Chapters:";

        const chapterContainer = document.getElementById("chapter-items-container");
        if (chapterContainer) {
            chapterContainer.innerHTML = "";
            const chList = isFlatStructure(activeWorkKey) ? GLOBAL_STRUCTURES[activeWorkKey] : GLOBAL_STRUCTURES[activeWorkKey][payload.book];
            chList.forEach(ch => {
                const a = document.createElement("a"); a.innerText = ch;
                if(ch === payload.chapter) a.className = "current";
                a.onclick = () => { activeSectionFilter = null; triggerTargetNavigation(payload.book, ch); };
                chapterContainer.appendChild(a);
            });
        }

        if (!isDramaOrPoetry) {
            const sectionContainer = document.getElementById("section-items-container");
            if (sectionContainer) {
                sectionContainer.innerHTML = "";
                Object.keys(payload.sections).forEach(sec => {
                    const a = document.createElement("a"); a.innerText = sec; a.className = "sec-pill";
                    if(sec === activeSectionFilter) a.classList.add("active-pill");
                    a.onclick = () => selectSectionDirectly(sec);
                    sectionContainer.appendChild(a);
                });
            }
        }

        const railNodesTarget = document.getElementById("rail-nodes-target");
        const railTitle = document.getElementById("rail-title");
        if (railNodesTarget) {
            railNodesTarget.innerHTML = "";
            if (isFlatStructure(activeWorkKey)) {
                railTitle.innerText = isDramaOrPoetry ? "Lines Tree" : "Chapters Tree";
                GLOBAL_STRUCTURES[activeWorkKey].forEach(ch => {
                    const n = document.createElement("div");
                    n.className = `rail-tree-item ${(payload.chapter === ch) ? 'active-rail-node' : ''}`;
                    n.innerText = isDramaOrPoetry ? `Lines ${ch}` : `Chapter ${ch}`;
                    n.onclick = () => { activeSectionFilter = null; triggerTargetNavigation(null, ch); };
                    railNodesTarget.appendChild(n);
                });
            } else {
                railTitle.innerText = `Book ${payload.book} Chapters`;
                GLOBAL_STRUCTURES[activeWorkKey][payload.book].forEach(ch => {
                    const n = document.createElement("div");
                    n.className = `rail-tree-item ${(payload.chapter === ch) ? 'active-rail-node' : ''}`;
                    n.innerText = `Chapter ${ch}`;
                    n.onclick = () => { activeSectionFilter = null; triggerTargetNavigation(payload.book, ch); };
                    railNodesTarget.appendChild(n);
                });
            }
        }
    }

    function renderActiveContentLayers(payload) {
        ['f', 'c1', 'c2', 'c3', 'c4'].forEach(prefix => {
            const targetContainer = document.getElementById(`content_${prefix}`);
            if (!targetContainer) return;
            targetContainer.innerHTML = "";
            
            const vId = columnEditions[prefix];
            const activeEditionMeta = TEXT_REGISTRY[vId];
            const cssClass = activeEditionMeta ? activeEditionMeta.class : "english-text";
            const shortId = activeEditionMeta ? activeEditionMeta.short_id : vId;
            const isPoetry = isPoetryWork(activeWorkKey);
            
            Object.keys(payload.sections).forEach(sec => {
                const isHidden = activeSectionFilter !== null && activeSectionFilter !== sec;
                const row = document.createElement("div");
                row.className = `section-row s-idx-${sec} ${isHidden ? 'hidden-section' : ''}`;
                const txt = payload.sections[sec][shortId] || "<i>[Text range missing in alignment layer]</i>";
                const visualIndexLabel = isPoetry ? sec : `[${sec}]`;
                
                if (isPoetry) {
                    const wrapper = document.createElement("div");
                    wrapper.className = `${cssClass} poetry-grid-layout`;
                    wrapper.innerHTML = txt;
                    row.appendChild(wrapper);
                } else {
                    row.innerHTML = `
                        <div class="prose-inline-layout ${cssClass}">
                            <span class="prose-marker"><a href="javascript:void(0)" onclick="selectSectionDirectly('${sec}')">${visualIndexLabel}</a></span>
                            <div class="prose-body-inline">${txt}</div>
                        </div>
                    `;
                }
                targetContainer.appendChild(row);
            });

            const footer = document.createElement("div");
            footer.className = "viewport-footer-controls";
            let prevBtnHtml = "", nextBtnHtml = "";
            if (payload.navigation.prev) {
                const prevParts = payload.navigation.prev.split(":");
                const prevPsg = prevParts[prevParts.length - 1].split(".");
                const label = isPoetry ? "Chapter" : (isFlatStructure(activeWorkKey) ? `Ch ${prevPsg[0]}` : `Bk ${prevPsg[0]} Ch ${prevPsg[1]}`);
                prevBtnHtml = `<a class="action-btn" onclick="navigateAdjacentUrn('${payload.navigation.prev}')">&larr; Previous (${label})</a>`;
            }
            if (payload.navigation.next) {
                const nextParts = payload.navigation.next.split(":");
                const nextPsg = nextParts[nextParts.length - 1].split(".");
                const label = isPoetry ? "Chapter" : (isFlatStructure(activeWorkKey) ? `Ch ${nextPsg[0]}` : `Bk ${nextPsg[0]} Ch ${nextPsg[1]}`);
                nextBtnHtml = `<a class="action-btn" onclick="navigateAdjacentUrn('${payload.navigation.next}')">Next (${label}) &rarr;</a>`;
            }
            
            footer.innerHTML = `
                <div class="footer-group-left">
                    ${prevBtnHtml}
                    <a href="javascript:void(0)" class="action-btn secondary" onclick="clearSectionFilter(event)">Full View</a>
                </div>
                ${nextBtnHtml}
            `;
            targetContainer.appendChild(footer);
        });
    }

    window.navigateAdjacentUrn = function(urn) {
        activeSectionFilter = null;
        const parts = urn.split(":");
        const psg = parts[parts.length - 1].split(".");
        if (isFlatStructure(activeWorkKey)) {
            triggerTargetNavigation(null, psg[0]);
        } else {
            triggerTargetNavigation(psg[0], psg[1]);
        }
    };

    window.clearSectionFilter = function(e) {
        if(e) e.preventDefault();
        activeSectionFilter = null;
        triggerViewRefresh();
    };

    window.updateColumnContent = function(prefix, value) {
        columnEditions[prefix] = value;
        triggerViewRefresh();
    };

    window.setWorkspaceLayoutMode = function(mode) {
        currentActiveMode = mode;
        const frame = document.getElementById("outer-wrapper");
        const btnParallel = document.getElementById("btn-mode-parallel");
        const btnClassic = document.getElementById("btn-mode-classic");
        
        if (mode === 'classic') {
            frame.className = "mode-classic";
            if (activeColumnsCount === 4) frame.classList.add('four-columns');
            if (activeColumnsCount === 5) frame.classList.add('five-columns');
            btnClassic.classList.add("active-mode"); btnParallel.classList.remove("active-mode");
        } else {
            frame.className = "mode-parallel";
            if (activeColumnsCount === 4) frame.classList.add('four-columns');
            if (activeColumnsCount === 5) frame.classList.add('five-columns');
            btnParallel.classList.add("active-mode"); btnClassic.classList.remove("active-mode");
        }
        
        updateClassicGridForColumns();
        triggerViewRefresh();
    };
  
    window.cycleColumns = function() {
        if (activeColumnsCount === 3) activeColumnsCount = 4;
        else if (activeColumnsCount === 4) activeColumnsCount = 5;
        else activeColumnsCount = 3;

        const btn = document.getElementById('btn-column-scaler');
        const ow = document.getElementById('outer-wrapper');
        
        ow.classList.remove('four-columns', 'five-columns');
        btn.innerText = `Columns: ${activeColumnsCount}`;
        
        if (activeColumnsCount === 4) ow.classList.add('four-columns');
        if (activeColumnsCount === 5) ow.classList.add('five-columns');
        
        updateClassicGridForColumns();
        triggerViewRefresh();
    };

    function updateClassicGridForColumns() {
        const ow = document.getElementById('outer-wrapper');
        const colF = document.getElementById('col_f');
        if (!ow || !ow.classList.contains('mode-classic')) return;
        
        if (activeColumnsCount === 5) colF.style.gridRow = '1 / span 4';
        else if (activeColumnsCount === 4) colF.style.gridRow = '1 / span 3';
        else colF.style.gridRow = '1 / span 2';
    }

    window.swapToFocus = function(sourcePrefix) {
        const focusEdition = columnEditions['f'];
        const sourceEdition = columnEditions[sourcePrefix];
        
        columnEditions['f'] = sourceEdition;
        columnEditions[sourcePrefix] = focusEdition;
        
        document.getElementById('select_f').value = sourceEdition;
        document.getElementById('select_' + sourcePrefix).value = focusEdition;
        
        triggerViewRefresh();
    };

    // ── Resizer Drag Framework ──────────────────────────────────────────
    (function() {
        function initResizeHandles() {
            document.querySelectorAll('.resize-handle').forEach(h => {
                h.addEventListener('mousedown', startResize);
                h.addEventListener('touchstart', startResize, {passive: false});
            });
        }

        function startResize(e) {
            e.preventDefault();
            const handle = e.currentTarget;
            const isHorizontal = handle.classList.contains('resize-handle-h');
            const container = document.getElementById('main-container');
            const isClassic = document.getElementById('outer-wrapper').classList.contains('mode-classic');

            handle.classList.add('dragging');
            document.body.classList.add(isHorizontal ? 'resizing-cols' : 'resizing-rows');

            if (isClassic) {
                handleClassicResize(e, handle, container);
            } else {
                handleParallelResize(e, handle, container);
            }
        }

        function handleParallelResize(startEvent, handle, container) {
            const columns = Array.from(container.querySelectorAll('.reading-column'));
            const handleIndex = Array.from(container.children).indexOf(handle);
            const prevCol = container.children[handleIndex - 1];
            const nextCol = container.children[handleIndex + 1];
            if (!prevCol || !nextCol) return;

            const startX = (startEvent.touches ? startEvent.touches[0] : startEvent).clientX;
            const prevStart = prevCol.getBoundingClientRect().width;
            const nextStart = nextCol.getBoundingClientRect().width;

            columns.forEach(col => { col.style.flex = 'none'; col.style.width = col.getBoundingClientRect().width + 'px'; });

            function onMove(e) {
                const clientX = (e.touches ? e.touches[0] : e).clientX;
                const dx = clientX - startX;
                prevCol.style.width = Math.max(100, prevStart + dx) + 'px';
                nextCol.style.width = Math.max(100, nextStart - dx) + 'px';
            }
            function onEnd() {
                handle.classList.remove('dragging'); document.body.classList.remove('resizing-cols', 'resizing-rows');
                document.removeEventListener('mousemove', onMove); document.removeEventListener('mouseup', onEnd);
            }
            document.addEventListener('mousemove', onMove); document.addEventListener('mouseup', onEnd);
        }

        function handleClassicResize(startEvent, handle, container) {
            const startX = (startEvent.touches ? startEvent.touches[0] : startEvent).clientX;
            const startY = (startEvent.touches ? startEvent.touches[0] : startEvent).clientY;
            const style = getComputedStyle(container);
            const colWidths = style.gridTemplateColumns.split(' ').map(parseFloat);
            const rowHeights = style.gridTemplateRows.split(' ').map(parseFloat);

            function onMove(e) {
                const clientX = (e.touches ? e.touches[0] : e).clientX;
                const clientY = (e.touches ? e.touches[0] : e).clientY;

                if (handle.id === 'resize-f-c1') {
                    const dx = clientX - startX; const totalW = colWidths[0] + colWidths[1];
                    const newLeft = Math.max(150, Math.min(totalW - 150, colWidths[0] + dx));
                    container.style.gridTemplateColumns = newLeft + 'px ' + (totalW - newLeft) + 'px';
                    positionClassicHandles();
                } else if (handle.id === 'resize-c1-c2') {
                    const dy = clientY - startY; const totalH = rowHeights[0] + rowHeights[1];
                    const newTop = Math.max(80, Math.min(totalH - 80, rowHeights[0] + dy));
                    container.style.gridTemplateRows = newTop + 'px ' + (totalH - newTop) + 'px';
                    positionClassicHandles();
                }
            }
            function onEnd() {
                handle.classList.remove('dragging'); document.body.classList.remove('resizing-cols', 'resizing-rows');
                document.removeEventListener('mousemove', onMove); document.removeEventListener('mouseup', onEnd);
            }
            document.addEventListener('mousemove', onMove); document.addEventListener('mouseup', onEnd);
        }

        function createClassicOverlayHandles() {
            const mc = document.getElementById('main-container');
            if (!mc) return;

            let vHandle = document.getElementById('classic-resize-col');
            if (!vHandle) {
                vHandle = document.createElement('div'); vHandle.id = 'classic-resize-col';
                vHandle.className = 'resize-handle resize-handle-h'; vHandle.style.cssText = 'position:absolute; top:0; bottom:0; z-index:25;';
                mc.style.position = 'relative'; mc.appendChild(vHandle);
                vHandle.addEventListener('mousedown', classicColResize);
            }
            let hHandle = document.getElementById('classic-resize-row');
            if (!hHandle) {
                hHandle = document.createElement('div'); hHandle.id = 'classic-resize-row';
                hHandle.className = 'resize-handle resize-handle-v'; hHandle.style.cssText = 'position:absolute; left:0; right:0; z-index:25;';
                mc.appendChild(hHandle); hHandle.addEventListener('mousedown', classicRowResize);
            }
            positionClassicHandles();
        }

        function positionClassicHandles() {
            const mc = document.getElementById('main-container');
            const colF = document.getElementById('col_f'); const colC1 = document.getElementById('col_c1');
            if (!mc || !colF || !colC1 || !document.getElementById('outer-wrapper')?.classList.contains('mode-classic')) return;

            const mcRect = mc.getBoundingClientRect(); const fRect = colF.getBoundingClientRect(); const c1Rect = colC1.getBoundingClientRect();
            if (document.getElementById('classic-resize-col')) document.getElementById('classic-resize-col').style.left = (fRect.right - mcRect.left - 2) + 'px';
            if (document.getElementById('classic-resize-row')) {
                document.getElementById('classic-resize-row').style.top = (c1Rect.bottom - mcRect.top - 2) + 'px';
                document.getElementById('classic-resize-row').style.left = (fRect.right - mcRect.left + 3) + 'px';
                document.getElementById('classic-resize-row').style.right = '0';
            }
        }

        function classicColResize(startEvent) {
            startEvent.preventDefault();
            const mc = document.getElementById('main-container'); const handle = document.getElementById('classic-resize-col');
            handle.classList.add('dragging'); document.body.classList.add('resizing-cols');
            const colWidths = getComputedStyle(mc).gridTemplateColumns.split(' ').map(parseFloat);
            const startX = startEvent.clientX;

            function onMove(e) {
                const dx = e.clientX - startX; const totalW = colWidths[0] + colWidths[1];
                const newLeft = Math.max(150, Math.min(totalW - 150, colWidths[0] + dx));
                mc.style.gridTemplateColumns = newLeft + 'px ' + (totalW - newLeft) + 'px';
                positionClassicHandles();
            }
            function onEnd() {
                handle.classList.remove('dragging'); document.body.classList.remove('resizing-cols');
                document.removeEventListener('mousemove', onMove); document.removeEventListener('mouseup', onEnd);
            }
            document.addEventListener('mousemove', onMove); document.addEventListener('mouseup', onEnd);
        }

        function classicRowResize(startEvent) {
            startEvent.preventDefault();
            const mc = document.getElementById('main-container'); const handle = document.getElementById('classic-resize-row');
            handle.classList.add('dragging'); document.body.classList.add('resizing-rows');
            const rowHeights = getComputedStyle(mc).gridTemplateRows.split(' ').map(parseFloat);
            const startY = startEvent.clientY;

            function onMove(e) {
                const dy = e.clientY - startY; const totalH = rowHeights[0] + rowHeights[1];
                const newTop = Math.max(80, Math.min(totalH - 80, rowHeights[0] + dy));
                mc.style.gridTemplateRows = newTop + 'px ' + (totalH - newTop) + 'px';
                positionClassicHandles();
            }
            function onEnd() {
                handle.classList.remove('dragging'); document.body.classList.remove('resizing-rows');
                document.removeEventListener('mousemove', onMove); document.removeEventListener('mouseup', onEnd);
            }
            document.addEventListener('mousemove', onMove); document.addEventListener('mouseup', onEnd);
        }

        const modeObserver = new MutationObserver(function() {
            const isClassic = document.getElementById('outer-wrapper')?.classList.contains('mode-classic');
            const h1 = document.getElementById('resize-f-c1'); const h2 = document.getElementById('resize-c1-c2'); 
            const h3 = document.getElementById('resize-c2-c3'); const h4 = document.getElementById('resize-c3-c4');
            const ch = document.getElementById('classic-resize-col'); const rh = document.getElementById('classic-resize-row');
            
            if (isClassic) {
                if (h1) h1.style.display = 'none'; if (h2) h2.style.display = 'none'; 
                if (h3) h3.style.display = 'none'; if (h4) h4.style.display = 'none';
                createClassicOverlayHandles();
                if (ch) ch.style.display = ''; if (rh) rh.style.display = '';
                setTimeout(positionClassicHandles, 50);
            } else {
                if (h1) h1.style.display = ''; if (h2) h2.style.display = ''; 
                if (h3) h3.style.display = activeColumnsCount >= 4 ? '' : 'none';
                if (h4) h4.style.display = activeColumnsCount === 5 ? '' : 'none';
                if (ch) ch.style.display = 'none'; if (rh) rh.style.display = 'none';
            }
        });

        document.addEventListener('DOMContentLoaded', function() {
            initResizeHandles();
            const ow = document.getElementById('outer-wrapper');
            if (ow) modeObserver.observe(ow, { attributes: true, attributeFilter: ['class'] });
            window.addEventListener('resize', positionClassicHandles);
        });
    })();

    const dz = document.getElementById('drop-zone');
    if(dz) {
        dz.addEventListener('dragover', e => { e.preventDefault(); dz.style.borderColor = '#660000'; dz.style.background = '#fdfbef'; });
        dz.addEventListener('dragleave', () => { dz.style.borderColor = '#ccc'; dz.style.background = '#fafafa'; });
        dz.addEventListener('drop', e => { e.preventDefault(); dz.style.borderColor = '#ccc'; dz.style.background = '#fafafa'; handleFileSelection(e.dataTransfer.files); });
    }
  </script>
</body>
</html>
""".replace("STRUCT_REPLACE", struct_map_json).replace("REGISTRY_REPLACE", text_registry_json)

(WORKSPACE_DIR / "index.html").write_text(INDEX_HTML_CONTENT, encoding='utf-8')
(WORKSPACE_DIR / ".nojekyll").write_text("", encoding='utf-8')
print("[SUCCESS] Production Standalone Workspace compiled cleanly.")

[SUCCESS] Production Standalone Workspace compiled cleanly.
